# Phishing Email Detection with Explainable AI (Colab GPU Runner)

This notebook executes the full production training pipeline on GPU in Google Colab:
1. GPU assertion (ensures CUDA environment)
2. Mount Google Drive & Set up persistent workspace
3. Install dependencies (`requirements-colab.txt` + `pip install -e .`)
4. Download & verify dataset (`scripts/download_data.py`) -> Drive backup
5. Clean dataset & generate leakage-free splits (`scripts/make_splits.py`) -> Drive backup
6. Hyperparameter tuning via random search (`scripts/run_tune.py`, skips if `best_params.json` exists) -> Drive backup
7. Final model training with optimal params (`scripts/run_train.py --params ... --resume --ckpt_dir ...`) -> Drive backup
8. Local explainability & LITA visualizations (`scripts/run_explain.py`) -> Drive backup
9. Faithfulness benchmarks: Comprehensiveness & Sufficiency (`scripts/run_faithfulness.py`) -> Drive backup
10. Alpha parameter ablation study on LITA (`scripts/run_ablation.py`) -> Drive backup
11. Consolidated report generation (`scripts/make_report.py`) -> Final Drive backup

In [ ]:
# Cell 1: Assert GPU availability
!nvidia-smi
import torch
assert torch.cuda.is_available(), "CUDA GPU is required for this Colab run. Please select a GPU runtime (Runtime -> Change runtime type -> T4/V100/A100)."
print(f"CUDA is active: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Mount Google Drive and navigate to project workspace
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/phishing_xai_results'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
os.makedirs('/content/checkpoints', exist_ok=True)

!mkdir -p /content/project
%cd /content/project
# If unzipping from Drive:
# !unzip -q /content/drive/MyDrive/Phishing_Email_Detection_XAI.zip -d /content/project/
# Or if cloning repository:
# !git clone <repo_url> .
%pwd
!ls -la

In [ ]:
# Cell 3: Install Colab-optimized dependencies
!pip install -q -r requirements-colab.txt
!pip install -q -e .
print("Dependencies successfully installed.")

In [ ]:
# Cell 4: Download and strictly verify dataset -> Backup to Drive
!python scripts/download_data.py

# Stage backup
!cp -r data /content/drive/MyDrive/phishing_xai_results/ || true
print("Dataset verified and backed up.")

In [ ]:
# Cell 5: Clean dataset, resolve conflicts, and create stratified splits -> Backup to Drive
!python scripts/make_splits.py

# Stage backup
!cp results/splits.json /content/drive/MyDrive/phishing_xai_results/
print("Splits generated and backed up.")

In [ ]:
# Cell 6: Hyperparameter tuning via random search (skips if best_params.json already exists)
import os
if not os.path.exists('results/tuning/best_params.json') and not os.path.exists('results/best_params.json'):
    !python scripts/run_tune.py \
        --data_path data/Phishing_Email.csv \
        --output_dir results \
        --n_iterations 10 \
        --max_length 256 \
        --subsample_train 4000 \
        --seed 42
else:
    print("Tuning results found; skipping re-tuning.")

# Stage backup
!cp -r results/tuning /content/drive/MyDrive/phishing_xai_results/ || true
print("Tuning stage completed and backed up.")

In [ ]:
# Cell 7: Fine-tune final model consuming optimal parameters on full dataset (with resume & ckpt_dir)
!python scripts/run_train.py \
    --data_path data/Phishing_Email.csv \
    --output_dir results \
    --params results/tuning/best_params.json \
    --ckpt_dir /content/checkpoints \
    --resume \
    --max_length 512 \
    --seed 42

# Stage backup
!cp results/*.pt /content/drive/MyDrive/phishing_xai_results/ || true
!cp results/*.json /content/drive/MyDrive/phishing_xai_results/ || true
!cp results/*.png /content/drive/MyDrive/phishing_xai_results/ || true
print("Training stage completed and backed up to Drive.")

In [ ]:
# Cell 8: Compute LIME, TI, and LITA local explanations and visualizations
!python scripts/run_explain.py \
    --checkpoint_path results/best_model.pt \
    --output_dir results/explanations \
    --num_samples 10 \
    --seed 42

# Stage backup
!cp -r results/explanations /content/drive/MyDrive/phishing_xai_results/ || true
print("Local explanations completed and backed up to Drive.")

In [ ]:
# Cell 9: Evaluate XAI Faithfulness (Comprehensiveness, Sufficiency, Random Baseline, Global Words)
!python scripts/run_faithfulness.py \
    --model_path results/best_model.pt \
    --output_dir results \
    --sample_size 100 \
    --max_length 256 \
    --seed 42

# Stage backup
!cp results/faithfulness* /content/drive/MyDrive/phishing_xai_results/ || true
!cp results/global_explanations.json /content/drive/MyDrive/phishing_xai_results/ || true
print("Faithfulness evaluation completed and backed up to Drive.")

In [ ]:
# Cell 10: Run Alpha Parameter Ablation Study on LITA (k=3 across alpha values and variants)
!python scripts/run_ablation.py \
    --model_path results/best_model.pt \
    --output_dir results \
    --sample_size 50 \
    --k 3 \
    --max_length 256 \
    --seed 42

# Stage backup
!cp results/alpha_ablation* /content/drive/MyDrive/phishing_xai_results/ || true
print("Alpha ablation completed and backed up to Drive.")

In [ ]:
# Cell 11: Generate consolidated markdown & JSON evaluation reports and final backup
!python scripts/make_report.py --results_dir results

# Final synchronization to Google Drive
!cp -r results/* /content/drive/MyDrive/phishing_xai_results/
print("Full pipeline complete! Displaying consolidated RESULTS.md:")
!cat results/RESULTS.md